# DS2002 · Visualization for Decisions

**Lecture — 2026-11-09 · Fall 2026**  
**Class time:** 45 minutes

---

## A chart is an argument

You have two presentations to give this semester. In both, the charts are doing the persuading, and most student charts fail in the same specific way: they report a topic instead of making a claim.

"Revenue by category" is a topic. "Food carries the night, but rain gear is where the money moves" is a claim. The chart is the same; the title is what tells the audience what they are looking at and why they should care.

Four rules for this course, and they are all enforced in the rubric:

1. **One chart, one message.** If it needs two sentences to explain, it is two charts.
2. **The title states the finding**, not the axes.
3. **Label the axes and name the unit.** "Revenue" is not a unit; "revenue (USD)" is.
4. **Sort by value** unless the category has a natural order like time.

Everything else today is in service of those four.

### Picking the chart from the question

There are more chart types than you need. Five cover almost everything in this course, and the question dictates the choice.

| The question | The chart |
|---|---|
| How do these categories compare? | Horizontal bar, sorted |
| How did this change over time? | Line, time on the x-axis |
| Do these two numbers move together? | Scatter |
| How is one number distributed? | Histogram |
| How do parts contribute to a whole over time? | Stacked area or grouped bar |

Pie charts are not on that list. Humans compare lengths well and angles badly, so a pie chart with more than three slices is a bar chart that lost information.

### Worked example — the default, then the fix

Start with what pandas gives you for free.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

revenue = pd.Series({'Food': 4200, 'Merch': 3100, 'RainGear': 900, 'Drink': 1800})
revenue.plot(kind='bar')
plt.show()

Everything is technically present and nothing is communicated. Four problems, and they are the four rules:

- No title, so the reader supplies their own interpretation.
- The y-axis says nothing. Dollars? Thousands? Units?
- Bars are in dictionary order, so comparing them takes effort.
- Vertical bars force the reader to tilt their head to read category names.

Same data, fixed:

In [ ]:
s = revenue.sort_values()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(s.index, s.values, color='#4a5d7e')
ax.set_title('Food drives game-day revenue; rain gear is the smallest line',
             loc='left', fontsize=12)
ax.set_xlabel('revenue (USD)')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
for y, v in enumerate(s.values):
    ax.text(v + 60, y, f'${v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

Sorted so the ranking is instant, horizontal so the labels read left to right, titled with the finding, unit named, and the values labeled so nobody has to squint at gridlines. Removing the top and right spines is not decoration — it takes ink off the page that was not carrying information.

### Worked example — time, and the annotation that carries the point

A line chart of orders by hour is fine. What makes it an argument is marking the moment that explains the shape.

In [ ]:
hours = np.arange(9, 22)
orders = np.array([10, 20, 35, 60, 90, 140, 220, 180, 120, 80, 60, 40, 25])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(hours, orders, marker='o', color='#4a5d7e')
ax.axvline(15.5, ls='--', color='gray', lw=1)
ax.annotate('kickoff 3:30pm', xy=(15.5, 225), xytext=(16.2, 215),
            fontsize=9, color='gray')
ax.set_title('Orders peak in the hour before kickoff, then fall off fast',
             loc='left', fontsize=12)
ax.set_xlabel('hour of day')
ax.set_ylabel('orders')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

The staffing decision falls straight out of that chart: put people on at 2pm, not 4pm. That is what "decision-ready" means — somebody can act without asking you a follow-up question.

### Four mistakes that will cost you points

**1. A truncated y-axis.** Starting a bar chart's axis above zero exaggerates differences. On a line chart it can be reasonable; on bars it is misleading, because the bar's *length* is the thing being compared.

In [ ]:
small = pd.Series({'Zone A': 980, 'Zone B': 940, 'Zone C': 910})

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(small.index, small.values, color='#b4553f')
axes[0].set_ylim(900, 1000)
axes[0].set_title('Axis starts at 900: looks like a 10x gap', fontsize=10)
axes[1].bar(small.index, small.values, color='#4a5d7e')
axes[1].set_ylim(0, 1000)
axes[1].set_title('Axis starts at 0: a 7% spread', fontsize=10)
plt.tight_layout()
plt.show()

**2. Overplotting.** Hundreds of points on a scatter turn into a blob. Transparency and smaller markers recover the shape.

In [ ]:
rng = np.random.default_rng(12)
x = rng.normal(20, 5, 900)
y = x * 2 + rng.normal(0, 8, 900)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].scatter(x, y)
axes[0].set_title('Default: a blob', fontsize=10)
axes[1].scatter(x, y, s=8, alpha=0.25, color='#4a5d7e')
axes[1].set_title('s=8, alpha=0.25: you can see density', fontsize=10)
for ax in axes:
    ax.set_xlabel('temperature (C)')
    ax.set_ylabel('ponchos sold')
plt.tight_layout()
plt.show()

**3. Comparing raw counts across different-sized groups.** Zone A has three times the foot traffic, so of course it sells more. Divide by the thing that makes them different and the chart starts answering the real question.

In [ ]:
zones = pd.DataFrame({
    'zone': ['A', 'B', 'C'],
    'revenue': [4800, 1900, 1200],
    'foot_traffic': [12000, 4000, 1500],
}).set_index('zone')
zones['revenue_per_visitor'] = zones['revenue'] / zones['foot_traffic']

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
zones['revenue'].plot(kind='bar', ax=axes[0], color='#4a5d7e')
axes[0].set_title('Zone A wins on total revenue', fontsize=10)
axes[0].set_ylabel('revenue (USD)')
zones['revenue_per_visitor'].plot(kind='bar', ax=axes[1], color='#b4553f')
axes[1].set_title('Zone C earns most per visitor', fontsize=10)
axes[1].set_ylabel('revenue per visitor (USD)')
plt.tight_layout()
plt.show()

Two opposite-looking conclusions from the same data, and both are true. Which one you show depends on the decision: total revenue for "where is our money," revenue per visitor for "where should we add a second cart."

**4. Forgetting `plt.tight_layout()`.** Labels get clipped, and a clipped chart in a presentation looks careless. Make it the last line before `plt.show()` every time.

### The four questions to ask before a chart goes in a deck

1. What is the one sentence this chart proves? If you cannot say it, the chart is not ready.
2. Is that sentence the title?
3. Can someone tell what the units are without asking?
4. Is anything on the page not carrying information?

Wednesday we run those questions against real charts, including some of yours.

### Practice 1 — one clean bar chart

Orders per zone: A is 900, B is 400, C is 250. Build a sorted horizontal bar chart with a title that states the finding, a labeled axis with the unit, and no top or right spine.

In [ ]:
# TODO

### Practice 2 — rewrite the titles

Turn each topic into a claim you could defend from the data above.

| Topic title | Your message title |
|---|---|
| "Revenue by category" | _..._ |
| "Orders by hour" | _..._ |
| "Revenue by zone" | _..._ |

### Practice 3 — fix a bad chart

The cell below has four problems from today's list. Identify them in a comment, then rewrite the chart.

In [ ]:
bad = pd.Series({'Drink': 1800, 'Food': 4200, 'RainGear': 900, 'Merch': 3100})
bad.plot(kind='bar')
plt.ylim(800, 4400)
plt.show()

# TODO: name the four problems, then rebuild it properly